# Week 2 — Using Pretrained Models with Hugging Face

**2026–2027 · revised 11 October 2026**

In Week 1, we learned how text becomes tokens, how attention adds context to their vectors, and how a model predicts the next token. This week, we load trained models in Python, inspect their inputs and outputs, and test a sentiment classifier on movie reviews.

1. Choose a model and load its tokenizer and trained weights.
2. Classify a review with a pipeline, then inspect the same calculation step by step.
3. Use BERT to fill a missing token and GPT-2 to continue **“A red toy robot”**.
4. Process several reviews together and measure processing speed and memory use.
5. Compare DistilBERT and BERT on the same labelled IMDB reviews.

By the end, you should be able to explain what each model receives, what its output means, and how to choose models for a given task.

Work together and use tools. Run cells in order; answer the short questions as you go. Private notes and results belong in `local/student/2026-2027/week_2/`, which Git ignores.


## Setup

Use the course environment from Week 1. If you need to create it, run these commands **in a terminal from the repository root**:

```bash
python3 -m venv .venv
source .venv/bin/activate  # Windows PowerShell: .\.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python -m ipykernel install --user --name llms-data-science --display-name "LLMs Data Science"
jupyter lab labs
```

Select that Python environment in Jupyter or VS Code.

These examples run locally on CPU and use public files: no API key or Hugging Face login is required. The first run needs internet access and disk space for model and dataset downloads; later runs normally reuse the downloaded files.

`torch` is PyTorch, the library doing the numerical calculations. `transformers` is the Python library for loading and using models; its name refers to the library, while **Transformer** in Week 1 referred to an architecture.


In [1]:
import torch
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

DEVICE = "cpu"  # The device where the model and its inputs will be placed.


### Explore the Hub

Open one model page and one dataset page. Read the model's task and usage example; preview a few rows of the dataset.

- [DistilBERT sentiment](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english): classify text as positive or negative.
- [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B): a small model trained for conversation.
- [Whisper](https://huggingface.co/openai/whisper-small): transcribe speech.
- [IMDB](https://huggingface.co/datasets/stanfordnlp/imdb): labelled movie reviews, used later in this lab.
- [Banking77](https://huggingface.co/datasets/PolyAI/banking77): customer queries in 77 banking categories.

Search [Models](https://huggingface.co/models) and [Datasets](https://huggingface.co/datasets) for a topic you care about. Which task could you try? You do not need to download these additional models.


## 1. Load a Tokenizer and a Trained Model

Our first task is sentiment classification. We choose the same DistilBERT sentiment model as in the lecture.

| Files | What we load |
|---|---|
| Tokenizer files | The rules and vocabulary that turn text into token IDs. |
| Trained weights | The numbers learned during training. A saved set of weights is a **checkpoint**. |
| Configuration | The settings needed to construct the model, including its output labels. |

A **version ID** identifies a particular set of files in this model repository. The loading functions call this argument `revision`. `HfApi().model_info(name).sha` looks up the current version ID, a Git commit hash.


In [2]:
name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
version_id = HfApi().model_info(name).sha
print("Model:", name)
print("Version ID:", version_id)


Model: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Version ID: 714eb0fa89d2f80546fda750413ed43d93601a13


Load the tokenizer and classifier from that **same version**. `AutoTokenizer` chooses the tokenizer specified by the files; `AutoModelForSequenceClassification` loads the model for assigning a label to a complete input sequence.

`use_safetensors=True` reads weights in the safetensors file format. `trust_remote_code=False` keeps the example in the installed library rather than executing extra Python code from the model repository. `.to(DEVICE)` places the model on our selected device, currently CPU.

Neither loading call makes a prediction. Looking up the current version again on another day may select newer files; reuse the recorded ID to select this version later.


In [3]:
# Load only the tokenizer: text <-> token IDs.
tokenizer = AutoTokenizer.from_pretrained(name, revision=version_id)

# Load the model: configuration + trained weights.
model = AutoModelForSequenceClassification.from_pretrained(
    name, revision=version_id,
    use_safetensors=True, trust_remote_code=False,
).to(DEVICE)
print("Output labels:", model.config.id2label)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Output labels: {0: 'NEGATIVE', 1: 'POSITIVE'}


## 2. Classify a Review, Then Inspect the Pipeline

A **pipeline** groups input preparation, the model calculation and interpretation of the output:

**text → tokenizer → token IDs → model → raw class scores → probabilities → label**

Running a trained model on an input is called **inference**. Its learned weights stay fixed. We give `pipeline()` the tokenizer and model already loaded above, so it reuses them.


In [4]:
classifier = pipeline(
    "text-classification", model=model, tokenizer=tokenizer, device=DEVICE,
)
text = "I loved this movie"
sentiment_result = classifier(text)
print(sentiment_result)


[{'label': 'POSITIVE', 'score': 0.9998757839202881}]


The result is a list containing one dictionary for our one review:

- `label` is the selected class: `NEGATIVE` or `POSITIVE`.
- `score` is the model probability of that class, calculated from the raw scores using softmax. We will inspect that calculation next.

A score of `0.953` does not establish a 95.3% chance that the sentence expresses positive sentiment.

### Try the limitation of two labels

This checkpoint has **no neutral output class**. Predict what it will return for each of these sentences before running the cell.


In [5]:
reviews = [
    "I love data!",
    "This is terrible...",
    "The actors were neither good nor bad, just average.",
]
for review, result in zip(reviews, classifier(reviews)):
    print(review, "->", result)


I love data! -> {'label': 'POSITIVE', 'score': 0.9998714923858643}
This is terrible... -> {'label': 'NEGATIVE', 'score': 0.9997186064720154}
The actors were neither good nor bad, just average. -> {'label': 'NEGATIVE', 'score': 0.9964141845703125}


### Step 1 — Turn the Text into Model Inputs

We now open the pipeline, starting with the **same review**.

`return_tensors="pt"` asks for PyTorch numerical arrays, called **tensors**, instead of ordinary Python lists. The tokenizer returns a dictionary:

- `input_ids`: one row of token IDs per input text.
- `attention_mask`: `1` for actual input tokens and `0` for padding. This single input needs no padding; we will use it for different-length reviews later.

`.to(DEVICE)` places these arrays on the same device as the model. `convert_ids_to_tokens()` lets us read the tokens corresponding to the IDs.


In [6]:
inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
print("Token IDs:", inputs["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist()))
print("Attention mask:", inputs["attention_mask"])


Token IDs: tensor([[ 101, 1045, 3866, 2023, 3185,  102]])
Tokens: ['[CLS]', 'i', 'loved', 'this', 'movie', '[SEP]']
Attention mask: tensor([[1, 1, 1, 1, 1, 1]])


### Step 2 — Run the Trained Model

`model.eval()` sets `training=False` on the model and its layers. For DistilBERT, this disables **dropout**, the random zeroing of activations used during training. Hugging Face already sets this mode when loading with `from_pretrained()`; the explicit call makes our intended mode clear.

`torch.inference_mode()` separately disables **gradient tracking**, the recording of operations used to calculate gradients for training. Neither operation changes the trained weights.

`**inputs` is Python dictionary unpacking: each dictionary entry becomes a named argument. Here, `model(**inputs)` is equivalent to:

```python
model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
```

It is not pointer syntax. The model returns an object containing `.logits`, the raw scores for the two classes.


In [7]:
model.eval()  # Set training=False; disable dropout.
with torch.inference_mode():  # No gradient tracking for this calculation.
    logits = model(**inputs).logits  # ** passes dictionary entries by name.
print("Raw scores:", logits)
print("Shape:", tuple(logits.shape))  # (number of reviews, number of classes)


Raw scores: tensor([[-4.3254,  4.6683]])
Shape: (1, 2)


### Step 3 — Turn Logits into Probabilities

The final classifier calculates a raw score, or **logit**, for each class:

$$
z_c = w_c^{\mathsf T}h + b_c.
$$

Here, $h$ is the review vector supplied to the final classifier; $w_c$ is the learned weight vector for class $c$, and $b_c$ is its learned bias. These logits can be negative and do not sum to one.

**Softmax** converts the logits into a probability vector:

$$
p_c = \frac{\exp(z_c)}{\exp(z_{\mathrm{NEGATIVE}}) + \exp(z_{\mathrm{POSITIVE}})}.
$$

Exponentiation makes the values positive; dividing by their sum makes the resulting probabilities sum to one. For example, logits $[-1, 2]$ give approximately $[0.047, 0.953]$. These are illustrative numbers, not a recorded prediction.

`dim=-1` applies softmax across the last dimension: the two classes. The first dimension contains reviews. `probabilities[0]` selects the **first review**, which is also the **only review** here; it does not select the first class.


In [8]:
probabilities = logits.softmax(dim=-1)
print("Probability rows:", probabilities)
print("First review's probabilities:", probabilities[0])
print("Sum for that review:", probabilities[0].sum().item())


Probability rows: tensor([[1.2417e-04, 9.9988e-01]])
First review's probabilities: tensor([1.2417e-04, 9.9988e-01])
Sum for that review: 0.9999999403953552


### Step 4 — Choose the Label

`argmax()` returns the position of the largest value. `.item()` converts a tensor containing one number to a Python number. `model.config.id2label` maps class positions to their label names.

We first print both classes, then choose the one with the larger probability. Compare this result with our original pipeline call.


In [9]:
for class_index, probability in enumerate(probabilities[0]):
    print(model.config.id2label[class_index], probability.item())

chosen_index = probabilities[0].argmax().item()
prediction = model.config.id2label[chosen_index]
print("Prediction:", prediction)
pipeline_result = classifier(text)  # Recompute if you changed the review.
print("Pipeline result:", pipeline_result)
assert prediction == pipeline_result[0]["label"]


NEGATIVE 0.00012416791287250817
POSITIVE 0.9998757839202881
Prediction: POSITIVE
Pipeline result: [{'label': 'POSITIVE', 'score': 0.9998757839202881}]


### Exercise — Explain Each Step

Change `text` to another short review and rerun Steps 1–4. What changes in the token IDs, logits and probabilities? Explain why the pipeline and the manual calculation select the same label.

Try negation or ambiguous wording. Does the larger probability by itself tell you whether the interpretation is correct?


## 3. A Different Task: Fill a Missing Token with BERT

The sentiment model chose between two labels. This BERT model instead scores **30,522 tokens** in its vocabulary, the tokenizer's fixed list of tokens. These include words, word pieces, punctuation and special tokens.

`[MASK]` marks the position to fill. BERT uses text on both sides to predict a token there. `top_k=5` asks this pipeline to show the five highest-scoring suggestions; the model scores the whole vocabulary first. Here `top_k` counts returned suggestions, not the generation sampling filter used later.


In [10]:
bert_name = "google-bert/bert-base-uncased"
bert_version_id = HfApi().model_info(bert_name).sha
fill_missing = pipeline(
    "fill-mask", model=bert_name, revision=bert_version_id, device=DEVICE,
)
suggestions = fill_missing("Paris is the [MASK] of France.", top_k=5)
for suggestion in suggestions:
    print(suggestion["token_str"], suggestion["score"])


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

capital 0.9969332218170166
heart 0.0005914860521443188
center 0.0004378740268293768
centre 0.0003378346737008542
city 0.0002699580800253898


### Expected Message When Loading BERT

You may see `Some weights ... were not used`. This checkpoint was trained for two tasks: predicting missing tokens and **next sentence prediction**, deciding whether one text segment followed another in the training text. Our `fill-mask` pipeline uses only the missing-token task. The listed `bert.pooler` and `cls.seq_relationship` weights belong to the other task and are unused here. This particular message is expected; the weights needed for missing-token prediction are loaded.

`Device set to use cpu` simply reports where the calculation runs. See the [BERT documentation](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertForPreTraining) for its two training tasks.

### Exercise — Change the Context

Try: `"Marseille is the real [MASK] of France."` Then change words before or after `[MASK]`. Which suggestions change?

Each returned `score` is the model probability of that token at the masked position. The five displayed probabilities need not sum to one: other vocabulary tokens also receive probability.


## 4. Generate Text: “A Red Toy Robot”

A **prompt** is the text we give the model. GPT-2 was trained on web text to predict the next token from the preceding tokens. To generate text, it chooses a next token, appends it and repeats—the loop from Week 1.

| Setting | Meaning in this call |
|---|---|
| `max_new_tokens=40` | Generate at most 40 new tokens beyond the prompt. |
| `do_sample=False` | Choose the highest-probability next token at each step: **greedy decoding**. |
| `return_full_text=False` | Return only the new text, without repeating the prompt. The model still receives the entire prompt as context. |

Keep the prompt **“A red toy robot”**: we can inspect how its context affects the continuation. First free the missing-token model before loading GPT-2.


In [11]:
if "fill_missing" in globals():
    del fill_missing  # Release BERT; this cell can also be rerun.
gpt2_name = "openai-community/gpt2"
gpt2_version_id = HfApi().model_info(gpt2_name).sha
generator = pipeline(
    "text-generation", model=gpt2_name, revision=gpt2_version_id, device=DEVICE,
)
prompt = "A red toy robot"
generation_result = generator(
    prompt, max_new_tokens=40, do_sample=False,
    temperature=1.0,  # Reset the sampling default; greedy decoding ignores it.
    pad_token_id=generator.tokenizer.eos_token_id,  # GPT-2's end-of-text token.
    return_full_text=False,
)
continuation = generation_result[0]["generated_text"]  # First and only result.
print("Prompt:", prompt)
print("New text:", continuation)
print("Prompt + new text:", prompt + continuation)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Prompt: A red toy robot
New text: , which is a robot that can be used to control a robot, is shown in the first episode of the first season.

The robot is shown in the first episode of the first season.
Prompt + new text: A red toy robot, which is a robot that can be used to control a robot, is shown in the first episode of the first season.

The robot is shown in the first episode of the first season.


### Read the Output

`Device set to use cpu` reports where the model runs.

`temperature=1.0` resets the pipeline's default sampling setting. Temperature only changes sampling; it has no effect when `do_sample=False`. Keeping a sampling temperature alongside greedy decoding can trigger the message that `temperature` may be ignored. We will use temperature for sampling below.

GPT-2 has no separate padding token. We explicitly use its end-of-text token ID, `50256`, as `pad_token_id`, instead of leaving the library to select it and print a message. Our single prompt is not padded; this does not add tokens to it.

`New text` contains only the generated tokens. `Prompt + new text` puts the original prompt back in front for reading. Repeated phrases and claims about episodes are generated text to inspect and verify, not evidence that those episodes exist.

For illustration, if the new text were `" walked across the table."`, `return_full_text=False` would return just that text. `True` would return `"A red toy robot walked across the table."` This setting changes the returned text, not the generation process.

### Choose Tokens by Sampling

**Sampling** means choosing a token randomly according to the model's probability distribution, rather than always taking the largest probability.

| Setting | What it changes |
|---|---|
| `do_sample=True` | Enable sampling. |
| `temperature` | A lower positive value concentrates probability on the more likely tokens; a higher value makes probabilities more even. |
| `top_p=0.9` | Keep the most likely tokens until their cumulative probability reaches at least 0.9; sample within that set after renormalizing. |
| `top_k=0` | Disable the additional filter that would retain only a fixed number of the most likely tokens. |

Keep the prompt, length limit and other filters fixed. Compare two temperatures and run two trials for each. `set_seed(42)` initializes the random generator for a repeatable starting point in the same environment; subsequent calls still draw new samples. Different wording is not evidence of greater factual accuracy.


In [12]:
from transformers import set_seed

set_seed(42)  # Set once, so successive trials still sample different tokens.
for temperature in [0.7, 1.2]:
    for trial in range(2):
        result = generator(
            prompt, max_new_tokens=40, do_sample=True,
            temperature=temperature, top_p=0.9, top_k=0,
            pad_token_id=generator.tokenizer.eos_token_id,
            return_full_text=False,
        )
        print(f"Temperature {temperature}, trial {trial + 1}:")
        print("Prompt:", prompt)
        print("New text:", result[0]["generated_text"], "\n")


Temperature 0.7, trial 1:
Prompt: A red toy robot
New text:  with a red eye is shown in this photo. (Source: Twitter)

It was also revealed that the robot is the first robot from the series to be launched into space.

In 

Temperature 0.7, trial 2:
Prompt: A red toy robot
New text:  is also shown to be running a single game on its own, and is even able to jump on a couple of people in the air.

In a news release, Nintendo said that a " 

Temperature 1.2, trial 1:
Prompt: A red toy robot
New text:  was activated by injection tests by subsequent student students from Caban Luna, the simulation simulation appeared to not be copying real arguments done by Seventh Generation Concepts projects also originating at SPS Vancouver. Lab equipment claimed 

Temperature 1.2, trial 2:
Prompt: A red toy robot
New text:  is attempting to chase a terrified cattle puppy and ensure the herded animal never rot on the fence," reported Padola, who isn't linked to Omensha in any way whatsoever.

Other 



### Exercise — Context and Randomness

Compare `"A red robot"`, `"A red toy robot"` and `"A blue toy robot"`. Use greedy decoding first, then repeat with sampling.

What changes when you add **toy**? What varies even when the prompt stays the same? Relate these observations to Week 1: context affects prediction; sampling introduces randomness; neither guarantees that a claim is true.


## 5. Devices, Batches and Padding

A **device** is where the numerical calculations run. This lab uses CPU by default. The model and its inputs must be on the same device.

| Hardware | PyTorch device setting |
|---|---|
| CPU | `"cpu"` |
| NVIDIA GPU with CUDA support | `"cuda"` |
| Supported AMD GPU with a ROCm build of PyTorch | Also `"cuda"`: ROCm reuses PyTorch's CUDA interface. |
| Compatible Mac GPU with MPS support | `"mps"` |

The right PyTorch installation depends on the hardware and operating system. [PyTorch installation](https://pytorch.org/get-started/locally/) and [AMD compatibility](https://rocm.docs.amd.com/en/latest/compatibility/compatibility-matrix.html) document the supported combinations.

If you try another available device, change `DEVICE` near the top and rerun from the start. Downloading successfully does not mean a model fits in the memory available to run it.


In [13]:
print("Selected device:", DEVICE)
print("NVIDIA / supported AMD GPU available:", torch.cuda.is_available())
print("Mac GPU available:", torch.backends.mps.is_available())


Selected device: cpu
NVIDIA / supported AMD GPU available: True
Mac GPU available: False


### Process Several Reviews Together

A **batch** is a group of inputs processed in one model call.

### Why Use Batches?

Suppose we need to classify **1,000 reviews**. Processing one review per call requires 1,000 model calls. With a batch size of 32, we need 32 calls: 31 full batches and one batch of eight reviews.

The model performs matrix calculations. Processing several reviews together gives it larger arrays to calculate with, which can use a GPU's parallel computing capacity more effectively and reduce the overhead of many separate calls. The aim is **more reviews processed per second**, called throughput.

Each review remains a separate input: reviews in the same batch do not supply context to one another, and batching does not train the model.

Larger batches need more working memory. Padding can also introduce extra computation, so a larger batch is not automatically faster. We will measure several batch sizes on the same reviews to see what works on our machine. Batching is useful when we already have many inputs to process; waiting to collect a batch can delay an individual interactive reply.

### Why Do We Need Padding?

Different-length token sequences need to form one rectangular array, so the tokenizer adds **padding** to the shorter ones.

- `padding=True`: pad to the longest sequence in this batch.
- `truncation=True, max_length=512`: keep at most 512 input tokens, including special tokens. This model cannot process longer inputs in one call.
- `attention_mask`: `1` for actual input tokens and `0` for padding. It is not a set of learned attention weights.

Inspect the IDs and masks below. How can you tell which positions are padding?


In [14]:
reviews = [
    "I absolutely loved this movie!",
    "The movie was okay; yet I cannot say anything positive about it.",
    "The plot was weak and boring.",
    "The actors were neither good nor bad, just average.",
]
review_inputs = tokenizer(
    reviews, padding=True, truncation=True, max_length=512, return_tensors="pt",
).to(DEVICE)
print("Token IDs:\n", review_inputs["input_ids"])
print("Padding mask:\n", review_inputs["attention_mask"])
print("Shape:", tuple(review_inputs["input_ids"].shape))  # (reviews, token positions)


Token IDs:
 tensor([[  101,  1045,  7078,  3866,  2023,  3185,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1996,  3185,  2001,  3100,  1025,  2664,  1045,  3685,  2360,
          2505,  3893,  2055,  2009,  1012,   102],
        [  101,  1996,  5436,  2001,  5410,  1998, 11771,  1012,   102,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1996,  5889,  2020,  4445,  2204,  4496,  2919,  1010,  2074,
          2779,  1012,   102,     0,     0,     0]])
Padding mask:
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])
Shape: (4, 16)


The classification calculation is unchanged. It now returns **four rows**, one per review, with two class scores in each row. We apply softmax to each row independently.

The pipeline can also process batches: `batch_size=4` tells it to process up to four reviews per call. Passing a list alone does not set this batch size.


In [15]:
with torch.inference_mode():
    batch_logits = model(**review_inputs).logits
batch_probabilities = batch_logits.softmax(dim=-1)
print("Probability rows:\n", batch_probabilities)
for review, row in zip(reviews, batch_probabilities):
    class_index = row.argmax().item()
    print(model.config.id2label[class_index], row[class_index].item(), "->", review)

print("Pipeline:", classifier(reviews, batch_size=4, truncation=True, max_length=512))


Probability rows:
 tensor([[1.2140e-04, 9.9988e-01],
        [8.2130e-03, 9.9179e-01],
        [9.9980e-01, 2.0337e-04],
        [9.9641e-01, 3.5858e-03]])
POSITIVE 0.9998785257339478 -> I absolutely loved this movie!
POSITIVE 0.9917870163917542 -> The movie was okay; yet I cannot say anything positive about it.
NEGATIVE 0.9997966885566711 -> The plot was weak and boring.
NEGATIVE 0.9964141845703125 -> The actors were neither good nor bad, just average.
Pipeline: [{'label': 'POSITIVE', 'score': 0.9998785257339478}, {'label': 'POSITIVE', 'score': 0.9917870163917542}, {'label': 'NEGATIVE', 'score': 0.9997966885566711}, {'label': 'NEGATIVE', 'score': 0.9964141845703125}]


## 6. Measure Batching on Labelled Movie Reviews

[IMDB](https://huggingface.co/datasets/stanfordnlp/imdb) supplies movie reviews with reference sentiment labels: **0 = negative, 1 = positive**. A **split** is one part of a dataset; we use `test`, which is intended for evaluation.

The classifier was trained for sentiment on **SST-2**, a different dataset. We are checking how it performs on these IMDB reviews, not reusing its training examples.

The `datasets` library loads the rows. As with the model, resolve one dataset version and record it. `dataset[0]` selects the first row; `dataset["label"]` selects the label column.


In [16]:
from datasets import concatenate_datasets, load_dataset

imdb_name = "stanfordnlp/imdb"
imdb_version_id = HfApi().dataset_info(imdb_name).sha
test_reviews = load_dataset(imdb_name, split="test", revision=imdb_version_id)
print("Dataset version:", imdb_version_id)
print("Label names:", test_reviews.features["label"].names)
print("First review:", test_reviews[0]["text"][:300])
print("Its reference label:", test_reviews[0]["label"])


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset version: e6281661ce1c48d982bc483cf8a173c1bbeb5d31
Label names: ['neg', 'pos']
First review: I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, C
Its reference label: 0


### Use the Same Sample for Every Measurement

Take **50 negative and 50 positive reviews**, then shuffle them. The first 100 rows alone might contain only one class. `shuffle(seed=42)` makes our selection repeatable for this dataset version; `select()` chooses row positions.

Add `row_id` before selecting, so we can record the exact test rows. All batch sizes and both models will use this same sample. Keep the input limit fixed at 512 tokens. Longer reviews are truncated, so the model may not see all of their text.


In [ ]:
PER_CLASS = 50
SEED = 42
MAX_LENGTH = 512

indexed_reviews = test_reviews.add_column("row_id", list(range(len(test_reviews))))
negative = indexed_reviews.filter(lambda row: row["label"] == 0).shuffle(seed=SEED).select(range(PER_CLASS))
positive = indexed_reviews.filter(lambda row: row["label"] == 1).shuffle(seed=SEED).select(range(PER_CLASS))
dataset = concatenate_datasets([negative, positive]).shuffle(seed=SEED)
print("Reviews:", len(dataset))
print("Negative:", list(dataset["label"]).count(0))
print("Positive:", list(dataset["label"]).count(1))


### What We Measure

$$
\text{accuracy} = \frac{\text{correct predictions}}{\text{number of reviews}},
\qquad
\text{throughput} = \frac{\text{number of reviews}}{\text{processing time in seconds}}.
$$

Compare batch sizes **1, 4, 16 and 32**. Time the same work: tokenization, transfer to the selected device, model calculation and collecting predictions. Loading and downloading are outside this measurement.

The following provided functions repeat steps you have already inspected. An untimed **warm-up** call prepares the device. GPU calculations are often **asynchronous**: Python sends work to the GPU and can continue to the next line while the GPU is still calculating. If we stop the timer then, the measured time may be too short. `synchronize()` makes Python wait until the GPU has finished all previously submitted work. We wait before starting the timer to finish the warm-up, and before stopping it to include the measured calculations.

Memory is reported separately:

- **CPU process RSS** (resident set size): memory resident in RAM for this Python process, sampled between batches. It includes more than this model and is not a continuous peak measurement.
- **GPU allocated memory**: PyTorch's peak tensor allocation on a CUDA/ROCm device during this run. This is not the CPU process memory, and we do not add the two. This helper does not measure MPS GPU memory.

`MB` here means one million bytes. Free the generation model before measuring the classifier.


In [ ]:
import gc
import time
import psutil
from sklearn.metrics import accuracy_score

# Remove the generator reference; keep the sentiment model for measurement.
del generator
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()  # Release unused GPU allocations cached by PyTorch.
elif DEVICE == "mps":
    torch.mps.empty_cache()


In [ ]:
def sync_device(device):
    kind = torch.device(device).type
    if kind == 'cuda': torch.cuda.synchronize(device)
    elif kind == 'mps': torch.mps.synchronize()


### Match Model Class IDs to Dataset Labels

Do not assume that position 0 always means negative. Read `model.config.id2label`, then map each model output position to IMDB's label IDs. The function below verifies the two sentiment names. If a checkpoint has generic names such as `LABEL_0`, it requires an explicit mapping from its documented training task.


In [ ]:
def imdb_label_map(model, explicit=None):
    if explicit is not None:
        mapping = {int(key): int(value) for key, value in explicit.items()}
    else:
        meanings = {'negative': 0, 'positive': 1}
        labels = {int(key): str(value).lower() for key, value in model.config.id2label.items()}
        if set(labels.values()) != set(meanings):
            raise ValueError('Verify label meanings in the model card and supply an explicit mapping.')
        mapping = {key: meanings[value] for key, value in labels.items()}
    if set(mapping) != set(range(model.config.num_labels)) or set(mapping.values()) != {0, 1}:
        raise ValueError('Expected a mapping from both classifier outputs to IMDB labels 0 and 1.')
    return mapping


### Provided Measurement Function

The function below makes an untimed call, processes all reviews in batches, and returns predictions plus the measurements defined above. The last batch can be smaller than `batch_size`.

Read the tokenizer → model → prediction lines first. The surrounding lines collect timing and memory measurements; they do not change the prediction method.


In [ ]:
def process_with_metrics(dataset, batch_size, model, tokenizer, label_to_imdb=None):
    if len(dataset) == 0 or batch_size < 1:
        raise ValueError('Use a non-empty dataset and positive batch size.')
    label_map = imdb_label_map(model, label_to_imdb)
    model.eval()
    warmup = tokenizer(dataset[:min(batch_size, len(dataset))]['text'], padding=True,
                       truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(model.device)
    with torch.inference_mode(): model(**warmup)
    sync_device(model.device)
    del warmup
    is_cuda = torch.device(model.device).type == 'cuda'
    if is_cuda: torch.cuda.reset_peak_memory_stats(model.device)
    process = psutil.Process()
    peak_cpu_mb = process.memory_info().rss / 1e6
    predictions = []
    started = time.perf_counter()
    for start in range(0, len(dataset), batch_size):
        encoded = tokenizer(dataset[start:start + batch_size]['text'], padding=True,
                            truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            logits = model(**encoded).logits
        predictions.extend(label_map[index] for index in logits.argmax(-1).cpu().tolist())
        peak_cpu_mb = max(peak_cpu_mb, process.memory_info().rss / 1e6)
    sync_device(model.device)
    seconds = time.perf_counter() - started
    return {'predictions': predictions,
            'accuracy': accuracy_score(dataset['label'], predictions),
            'seconds': seconds, 'reviews_per_second': len(dataset) / seconds,
            'sampled_peak_cpu_rss_mb': peak_cpu_mb,
            'peak_gpu_allocated_mb': torch.cuda.max_memory_allocated(model.device) / 1e6 if is_cuda else None}


### Run and Plot the Comparison of Batch Sizes

Each call uses the same model, reviews and input limit. Inspect any changed predictions: batching should preserve the task, but numerical differences can affect a prediction near a decision boundary.

A larger batch may be faster or slower on your machine. Choose from the measured results rather than assuming a GPU or a larger batch improves performance.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

batch_sizes = [1, 4, 16, 32]
results = {size: process_with_metrics(dataset, size, model, tokenizer) for size in batch_sizes}
batch_table = pd.DataFrame([
    {"batch_size": size, **{key: value for key, value in result.items() if key != "predictions"}}
    for size, result in results.items()
])
display(batch_table)
for size, result in results.items():
    changed = sum(a != b for a, b in zip(result["predictions"], results[1]["predictions"]))
    print(f"Batch size {size}: {changed} predictions differ from batch size 1.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(batch_table["batch_size"], batch_table["reviews_per_second"], marker="o")
axes[0].set(xlabel="Batch size", ylabel="Reviews per second", title="Processing speed")
axes[1].plot(batch_table["batch_size"], batch_table["sampled_peak_cpu_rss_mb"], marker="o")
axes[1].set(xlabel="Batch size", ylabel="CPU process RSS (MB)", title="Sampled CPU memory")
plt.tight_layout()
plt.show()


### Exercise — Read the Measurements

Which batch size gave the highest throughput? Did its accuracy change? What happened to memory use?

Repeat one measurement before drawing a conclusion from a small runtime difference. Changing batch size does not train the model. Explain why CPU process RSS is not the size of its weights.


## 7. Lab Exercise: Compare DistilBERT and BERT

We have measured DistilBERT. Now repeat the same calculation with the original [BERT SST-2 sentiment checkpoint](https://huggingface.co/textattack/bert-base-uncased-SST-2).

Use the same 100 reviews, batch size 4, device and 512-token limit. Both checkpoints were trained for SST-2 sentiment classification; a difference here is not an isolated experiment on architecture or model size, since their training also differs.

First record DistilBERT's result and parameter count, then free it before loading BERT. A **parameter** is a learned model value; the count describes model size, not total runtime memory.


In [ ]:
COMPARISON_BATCH_SIZE = 4
comparison = {"DistilBERT": results[COMPARISON_BATCH_SIZE]}
parameter_counts = {"DistilBERT": sum(parameter.numel() for parameter in model.parameters())}  # numel() counts array entries.

# classifier and model refer to the same weights; remove both references.
del classifier, model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()


### Load BERT and Check Its Labels

This is a **different BERT checkpoint** from the missing-token model: it has been trained to classify sentiment. Load its tokenizer and classifier from the same version, as in Section 1.

Its [configuration](https://huggingface.co/textattack/bert-base-uncased-SST-2/blob/6746dde1c9f5e0a76e4c3428ffa499ea2ecb49bc/config.json) identifies the SST-2 task. That task's labels are [0 = negative, 1 = positive](https://huggingface.co/datasets/stanfordnlp/sst2). The checkpoint may display generic names `LABEL_0` and `LABEL_1`, so we supply the task's mapping explicitly rather than interpreting those strings as sentiment names. Verify this ordering before substituting another checkpoint.


In [ ]:
comparison_name = "textattack/bert-base-uncased-SST-2"
comparison_version_id = HfApi().model_info(comparison_name).sha
comparison_tokenizer = AutoTokenizer.from_pretrained(comparison_name, revision=comparison_version_id)
comparison_model = AutoModelForSequenceClassification.from_pretrained(
    comparison_name, revision=comparison_version_id,
).to(DEVICE)
print("Version ID:", comparison_version_id)
print("Configured labels:", comparison_model.config.id2label)

# This checkpoint uses SST-2's class order: negative=0, positive=1.
bert_to_imdb = {0: 0, 1: 1}  # Model class ID -> IMDB label ID.
comparison["BERT"] = process_with_metrics(
    dataset, COMPARISON_BATCH_SIZE, comparison_model, comparison_tokenizer,
    label_to_imdb=bert_to_imdb,
)
parameter_counts["BERT"] = sum(parameter.numel() for parameter in comparison_model.parameters())


### Compare Quality and Runtime

The table uses the **same reviews** for both models. Accuracy measures agreement with the reference labels. Throughput measures processing speed on this device. Neither number describes performance on every dataset.

CPU process memory can retain allocations from previous calls; treat it as a process measurement, not a clean comparison of model weight sizes.


In [ ]:
comparison_table = pd.DataFrame([
    {"model": model_name, "parameters": parameter_counts[model_name],
     **{key: value for key, value in metrics.items() if key != "predictions"}}
    for model_name, metrics in comparison.items()
])
display(comparison_table)


### Inspect Mistakes Before Choosing

For each model, inspect up to one misclassified review from each reference class. Read its full text: the classifier sees at most the first 512 tokens, so relevant information may be missing from its input.

Is the wording ambiguous? Does the review mix praise and criticism? These are hypotheses to investigate, not established explanations of the model's error.


In [ ]:
for model_name, metrics in comparison.items():
    print("\n", model_name)
    for reference_label in [0, 1]:
        for row, predicted_label in zip(dataset, metrics["predictions"]):
            if row["label"] == reference_label and predicted_label != reference_label:
                print("Row:", row["row_id"], "reference:", row["label"], "prediction:", predicted_label)
                print(row["text"], "\n")
                break
        else:
            print("No mistake for reference class", reference_label, "in this sample.")


### Make and Explain a Choice

Which classifier would you use for these movie reviews, and why? Support your choice with the measured accuracy, mistakes and processing speed. What additional examples would you test before relying on that choice?

To extend the comparison, change `PER_CLASS`, rerun from the dataset selection, and keep the same rows for both models. If you tune a model or choose settings from data, use a separate validation set; keep test data for the final evaluation.

There is no submission requirement. Keep any private notes and exported results in the local directory below.


### Keep a Local Record

Record the model and dataset version IDs, selected rows, library versions, device, input limit and measurements. This lets you explain what was actually compared. The next cell saves that record under the ignored local student directory, whether Jupyter was started from the repository root or `labs/`.


In [ ]:
import json
from importlib.metadata import version
from pathlib import Path

course_root = Path.cwd()
if course_root.name == "labs":
    course_root = course_root.parent
if not (course_root / "labs" / "week_2_huggingface_basics.ipynb").is_file():
    raise FileNotFoundError("Start this notebook from the repository root or labs directory.")
work_dir = course_root / "local" / "student" / "2026-2027" / "week_2"
work_dir.mkdir(parents=True, exist_ok=True)
record = {
    "models": {name: version_id, comparison_name: comparison_version_id},
    "demonstration_models": {bert_name: bert_version_id, gpt2_name: gpt2_version_id},
    "dataset": imdb_name, "dataset_version_id": imdb_version_id,
    "test_row_ids": list(dataset["row_id"]), "seed": SEED,
    "libraries": {package: version(package) for package in
                  ["torch", "transformers", "huggingface-hub", "datasets", "scikit-learn", "psutil"]},
    "device": DEVICE, "max_length": MAX_LENGTH,
    "batch_sizes": batch_sizes, "batch_results": results,
    "comparison_batch_size": COMPARISON_BATCH_SIZE, "comparison": comparison,
    "parameter_counts": parameter_counts,
}
record_path = work_dir / "model_comparison.json"
record_path.write_text(json.dumps(record, indent=2) + "\n")
print("Saved locally:", record_path)


## Further Reading

- [Hugging Face: Behind the pipeline](https://huggingface.co/learn/llm-course/chapter2/2) follows text → tokenizer → model scores → probabilities.
- [Pipeline tutorial](https://huggingface.co/docs/transformers/en/pipeline_tutorial) covers task calls and batching.
- [GPT-2 model card](https://huggingface.co/openai-community/gpt2) describes its next-token training and intended uses.
- [PyTorch evaluation mode](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval) and [inference mode](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html) explain the two settings used in the manual calculation.
- [PyTorch on AMD](https://docs.pytorch.org/docs/stable/notes/hip.html) explains why ROCm uses the `cuda` device name.

The models in this lab are small teaching examples. The steps we inspected—providing context, interpreting outputs and checking results—also matter when we later put a model inside an agent workflow.
